# FIDO Task 1 — Ground-Truth Sanity Check

This notebook overlays the tissue **target point** — derived from the segmented
instrument tip plus the Task 1 ground-truth tool-tissue distance (`z`) — onto the
**ILM segmentation** for a given B-scan. If the ground-truth distance is scaled
correctly, the target point (blue) should line up with the ILM surface (yellow).

**Background:** while preparing to compute the Distance AUC for Task 1, this
check revealed that the target point did *not* align with the ILM surface. 

**Usage:** set `path` in the *Visualization* section below to your local FIDO
Task 1 dataset folder and run all cells.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from enum import Enum
import json

# Helper Functions 

Main Code is below this section

In [ ]:
class GenericLabels(Enum):
    Background = 0
    Ilm = 1
    Rpe = 2
    IlmBack = 20
    ArteriesOrVeins = 3
    Arteries = 4
    Veins = 5
    Liquid = 7
    Forceps = 8
    Cannula = 9
    Endoilluminator = 10
    InstrumentInOCT = 11
    ToolMirrorOCTArtifact = 12
    OCTUncapturedArea = 14

INSTRUMENT_LABELS = (
    GenericLabels.Forceps.value,
    GenericLabels.Cannula.value,
    GenericLabels.Endoilluminator.value,
    GenericLabels.InstrumentInOCT.value,
)

In [ ]:
def lowest_x_instrument_point(seg: np.ndarray):
    """Pixel (x, y) of the instrument-labeled pixel with the smallest x, or None.

    Considers Forceps / Cannula / Endoilluminator / InstrumentInOCT. Ties on x
    are broken by the smallest y (topmost).
    """
    if seg is None:
        return None
    mask = np.isin(seg, INSTRUMENT_LABELS)
    ys, xs = np.where(mask)
    if xs.size == 0:
        return None
    min_x = xs.min()
    y = ys[xs == min_x].min()
    return int(min_x), int(y)

In [ ]:
GROUND_TRUTH_DISTANCE_SCALE = 10 # or 4000/512 

def load_bscan_target(scenario: str, frame_id: str, root: Path, bscan_idx: int = 0):
    """
    Load one B-scan + its segmentation + the numerical ground truth for a sample,
    locate the instrument tip via `lowest_x_instrument_point`, and derive the tissue
    target point lying `tool_tissue_distance` pixels above the tip on the B-scan.

    Returns:
        bscan (np.ndarray), seg (np.ndarray),
        toolpoint (x, y) | None, target_point (x, y) | None
    """
    bscan_dir = root / scenario / "iOCT Microscope" / "Bscan" / frame_id
    bscan = np.array(Image.open(bscan_dir / f"{bscan_idx:02d}.png"))
    seg = np.array(Image.open(bscan_dir / "Segmentation" / f"{bscan_idx:02d}.png"))

    with open(root / scenario / "Numerical" / f"{frame_id}.json", "r") as f:
        numerical = json.load(f)
    _, _, z = numerical["Ground Truth"]["Task 1"]
    tool_tissue_distance = z / GROUND_TRUTH_DISTANCE_SCALE

    toolpoint = lowest_x_instrument_point(seg)
    if toolpoint is None:
        return bscan, seg, None, None

    tx, ty = toolpoint
    target_point = (tx, ty + tool_tissue_distance)
    return bscan, seg, toolpoint, target_point

# Visualization 

Required: Set the path to the dataset folder of task 1


**Expected result after the depth-resolution fix:** the blue target point
should align with the yellow ILM boundary.

In [ ]:
path = Path(".../Task 1") # required to be set to the path of the FIDO dataset (task 1) on your machine

idx = "00200"
scenario = "Scenario_01"

for bscan_idx in (0, 1):
    bscan_0, seg_0, toolpoint_0, target_point_0 = load_bscan_target(scenario, str(idx), path, bscan_idx)
    #bscan_1, seg_1, toolpoint_1, target_point_1 = load_bscan_target(scenario, str(idx), path, 1)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(bscan_0, cmap="gray")

    # red RGBA overlay layer: only visible (alpha>0) where the segmentation is > 0
    seg_overlay = np.zeros((*seg_0.shape, 4))
    seg_overlay[..., 0] = 1.0  # red channel
    seg_overlay[..., 3] = (seg_0 == GenericLabels.InstrumentInOCT.value).astype(float) * 0.6  # alpha only at segmented pixels

    # yellow RGBA overlay layer for the ILM surface boundary
    ilm_overlay = np.zeros((*bscan_0.shape, 4))
    ilm_overlay[..., 0] = 1.0  # red channel
    ilm_overlay[..., 1] = 1.0  # green channel -> combined yellow
    ilm_overlay[..., 3] = (seg_0 == GenericLabels.Ilm.value).astype(float) * 0.9

    # Tooltip
    x,y = lowest_x_instrument_point(seg_0) # type: ignore

    ax.imshow(seg_overlay)
    ax.imshow(ilm_overlay)
    ax.set_title(f"{scenario} | {idx} — Bscan - {bscan_idx}")
    ax.plot(x, y, "go", markersize=10, label="Lowest-y instrument point")
    ax.plot(*target_point_0, "bo", markersize=10, label="Target point (tool-tissue distance)")
    ax.plot([], [], color="yellow", linewidth=4, label="ILM surface boundary")
    ax.axis("off")
    plt.tight_layout()
    ax.legend()
    plt.show()
